# LLM UI Element Selection Experiment
## Query Ambiguity × Interface Complexity — Colab Runner
**Models**: Phi-3 Mini · Gemma 2 2B · TinyLlama · Mistral 7B · Llama 3 8B  
**Dataset**: 360 rows · 60 tasks · GitHub + AWS  
**Runtime**: Use GPU (Runtime → Change runtime type → T4 GPU) for best speed

In [ ]:
# ── CELL 1: Install everything ──────────────────────────────────────────────
# Run this first. Takes about 2 minutes.
!pip install -q ollama pandas openpyxl matplotlib seaborn scipy groq
!curl -fsSL https://ollama.com/install.sh | sh
print("✓ Installation complete")

In [ ]:
# ── CELL 2: Start Ollama service ────────────────────────────────────────────
import subprocess, time, requests

# Start Ollama in background
proc = subprocess.Popen(["ollama","serve"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)

# Verify it is running
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    print("✓ Ollama is running")
    print("  Available models:", [m["name"] for m in r.json().get("models",[])])
except:
    print("✗ Ollama not responding — re-run this cell")

In [ ]:
# ── CELL 3: Pull models ──────────────────────────────────────────────────────
# This downloads the models once. Takes 5-15 min depending on Colab speed.
# Comment out any models you do not want.

MODELS_TO_PULL = [
    "phi3:mini",    # 2.3 GB — Microsoft Phi-3 Mini (recommended)
    "gemma2:2b",    # 1.6 GB — Google Gemma 2
    "tinyllama",    # 0.6 GB — TinyLlama 1.1B
    "mistral",      # 4.1 GB — Mistral 7B  (needs T4 GPU)
    "llama3",       # 4.7 GB — Llama 3 8B  (needs T4 GPU)
]

for model in MODELS_TO_PULL:
    print(f"Pulling {model}...")
    result = subprocess.run(["ollama","pull",model], capture_output=True, text=True, timeout=600)
    if result.returncode == 0:
        print(f"  ✓ {model} ready")
    else:
        print(f"  ✗ {model} failed: {result.stderr[:100]}")

print("\nAll pulls complete!")

In [ ]:
# ── CELL 4: Upload your dataset ──────────────────────────────────────────────
from google.colab import files
import pandas as pd

print("Click 'Choose Files' and upload: llm_ui_selection_dataset_CORRECTED.xlsx")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
df = pd.read_excel(filename)
print(f"\n✓ Dataset loaded: {len(df)} rows, {df['task_id'].nunique()} tasks")
print(f"  Columns: {df.columns.tolist()}")
print(f"  Sample row:")
print(f"  {df.iloc[0].to_dict()}")

In [ ]:
# ── CELL 5: Configure models ─────────────────────────────────────────────────
# Edit this list to match the models you pulled in Cell 3

MODELS = {
    "phi3":      {"ollama_name": "phi3:mini",  "display": "Phi-3 Mini",  "color": "#e74c3c"},
    "gemma2":    {"ollama_name": "gemma2:2b",  "display": "Gemma 2 2B",  "color": "#3498db"},
    "tinyllama": {"ollama_name": "tinyllama",  "display": "TinyLlama",   "color": "#2ecc71"},
    "mistral":   {"ollama_name": "mistral",    "display": "Mistral 7B",  "color": "#9b59b6"},
    "llama3":    {"ollama_name": "llama3",     "display": "Llama 3 8B",  "color": "#f39c12"},
}

SYSTEM_PROMPT = (
    "You are a UI element selection assistant. "
    "You will be given a user query and a numbered list of UI elements visible on a screen. "
    "Select EXACTLY ONE UI element from the list that best matches the user query.\n\n"
    "Rules:\n"
    "- Respond with ONLY the exact element name — nothing else.\n"
    "- Do NOT add explanation, numbering, punctuation, or extra words.\n"
    "- The name must match exactly as given in the list."
)

def build_prompt(row):
    els = [e.strip() for e in str(row['ui_elements']).split(';')]
    numbered = '\n'.join(f'{i+1}. {e}' for i, e in enumerate(els))
    return (
        f"Platform: {row['platform']}\n"
        f"Interface: {row['interface_size']} ({len(els)} elements)\n"
        f"User Query: \"{row['query']}\"\n\n"
        f"UI Elements:\n{numbered}\n\n"
        f"Reply with ONLY the exact element name."
    )

import re
def clean_prediction(raw, elements):
    if not raw or raw in ("ERROR","TIMEOUT"): return raw
    raw = raw.strip().strip('"').strip("'")
    raw = re.sub(r'^\d+[\.)\s]+', '', raw)
    for el in elements:
        if el.strip().lower() == raw.lower(): return el.strip()
    for el in elements:
        if el.strip().lower() in raw.lower(): return el.strip()
    for el in elements:
        if raw.lower() in el.strip().lower(): return el.strip()
    return raw

def is_correct(pred, correct):
    if not pred or pred in ("ERROR","TIMEOUT"): return False
    return pred.strip().lower() == correct.strip().lower()

print("✓ Configuration ready")
print(f"  Models configured: {[MODELS[m]['display'] for m in MODELS]}")

In [ ]:
# ── CELL 6: Run experiment ───────────────────────────────────────────────────
# This runs ALL models on ALL 360 rows automatically.
# Progress is shown in real time.

import requests as req_lib, time, pandas as pd
from IPython.display import clear_output

OLLAMA_URL = "http://localhost:11434/api/generate"

def query_ollama(model_name, prompt, timeout=120):
    payload = {
        "model": model_name, "prompt": prompt, "stream": False,
        "options": {"temperature": 0, "num_predict": 30}
    }
    try:
        resp = req_lib.post(OLLAMA_URL, json=payload, timeout=timeout)
        resp.raise_for_status()
        return resp.json().get("response","").strip()
    except req_lib.exceptions.Timeout:
        return "TIMEOUT"
    except req_lib.exceptions.ConnectionError:
        return "OLLAMA_NOT_RUNNING"
    except Exception as e:
        return f"ERROR:{str(e)[:40]}"

all_results = {}

for mk, cfg in MODELS.items():
    print(f"\n{'='*55}")
    print(f"  Running: {cfg['display']}  ({cfg['ollama_name']})")
    print(f"{'='*55}")
    records = []
    correct_count = 0

    for i, (_, row) in enumerate(df.iterrows()):
        prompt   = build_prompt(row)
        elements = [e.strip() for e in str(row['ui_elements']).split(';')]
        raw      = query_ollama(cfg["ollama_name"], prompt)
        pred     = clean_prediction(raw, elements)
        ok       = is_correct(pred, row['correct_answer'])
        if ok: correct_count += 1
        records.append({
            "task_id":         row["task_id"],
            "platform":        row["platform"],
            "query":           row["query"],
            "ambiguity_level": row["ambiguity_level"],
            "interface_size":  row["interface_size"],
            "ui_elements":     row["ui_elements"],
            "correct_answer":  row["correct_answer"],
            "predicted":       pred,
            "is_correct":      ok,
            "model":           cfg["display"],
        })

        if (i+1) % 30 == 0 or not ok:
            pct = 100*correct_count/(i+1)
            status = "OK" if ok else "XX"
            print(f"  [{status}] {i+1:3d}/{len(df)} | {row['ambiguity_level']:12s} | "
                  f"{row['interface_size']:5s} | acc={pct:.1f}% | {str(pred)[:30]}")
        time.sleep(0.05)

    result_df = pd.DataFrame(records)
    all_results[mk] = result_df
    acc = result_df['is_correct'].mean()*100
    print(f"\n  DONE: {cfg['display']} — Accuracy: {correct_count}/{len(df)} = {acc:.1f}%")

print("\n✓ All models completed!")

In [ ]:
# ── CELL 7: Compute statistics ───────────────────────────────────────────────
import numpy as np

def compute_stats(df):
    s = {}
    s["overall"] = {"correct":int(df["is_correct"].sum()),"total":len(df),
                    "pct":round(100*df["is_correct"].mean(),2)}
    s["by_ambiguity"] = {}
    for lv in ["clear","paraphrased","ambiguous"]:
        sub = df[df["ambiguity_level"]==lv]
        s["by_ambiguity"][lv] = {"correct":int(sub["is_correct"].sum()),"total":len(sub),
                                  "pct":round(100*sub["is_correct"].mean(),2) if len(sub)>0 else 0}
    s["by_size"] = {}
    for sz in ["small","large"]:
        sub = df[df["interface_size"]==sz]
        s["by_size"][sz] = {"correct":int(sub["is_correct"].sum()),"total":len(sub),
                             "pct":round(100*sub["is_correct"].mean(),2) if len(sub)>0 else 0}
    s["by_platform"] = {}
    for pl in ["GitHub","AWS"]:
        sub = df[df["platform"]==pl]
        s["by_platform"][pl] = {"correct":int(sub["is_correct"].sum()),"total":len(sub),
                                  "pct":round(100*sub["is_correct"].mean(),2) if len(sub)>0 else 0}
    s["cross"] = {}
    for lv in ["clear","paraphrased","ambiguous"]:
        s["cross"][lv] = {}
        for sz in ["small","large"]:
            sub = df[(df["ambiguity_level"]==lv)&(df["interface_size"]==sz)]
            s["cross"][lv][sz] = {"correct":int(sub["is_correct"].sum()),"total":len(sub),
                                   "pct":round(100*sub["is_correct"].mean(),2) if len(sub)>0 else 0}
    return s

all_stats = {mk: compute_stats(all_results[mk]) for mk in all_results}

# Print summary table
print("="*60)
print("  RESULTS SUMMARY")
print("="*60)
print(f"  {'Model':<16} {'Acc%':>6}  {'Correct':>8}  {'Clear%':>7}  {'Para%':>7}  {'Ambig%':>7}  {'Small%':>7}  {'Large%':>7}")
print("  "+"-"*75)
for mk in all_stats:
    s = all_stats[mk]
    print(f"  {MODELS[mk]['display']:<16} {s['overall']['pct']:>6.1f}  "
          f"{s['overall']['correct']:>3}/{s['overall']['total']:<4}  "
          f"{s['by_ambiguity']['clear']['pct']:>6.1f}%  "
          f"{s['by_ambiguity']['paraphrased']['pct']:>6.1f}%  "
          f"{s['by_ambiguity']['ambiguous']['pct']:>6.1f}%  "
          f"{s['by_size']['small']['pct']:>6.1f}%  "
          f"{s['by_size']['large']['pct']:>6.1f}%")
print("  "+"-"*75)
print(f"  {'Random baseline':<16} {'~7%':>6}  {'':>8}  {'~10%':>7}  {'~10%':>7}  {'~10%':>7}  {'~10%':>7}  {'~4.5%':>7}")

In [ ]:
# ── CELL 8: Generate all charts ──────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import os
os.makedirs("charts", exist_ok=True)
os.makedirs("results", exist_ok=True)

plt.rcParams.update({
    "figure.facecolor":"#0d1117","axes.facecolor":"#161b22","axes.edgecolor":"#30363d",
    "axes.labelcolor":"#c9d1d9","axes.titlecolor":"#f0f6fc","xtick.color":"#8b949e",
    "ytick.color":"#8b949e","text.color":"#c9d1d9","grid.color":"#21262d","grid.alpha":0.5,
})

def save_chart(fig, name):
    fig.savefig(f"charts/{name}", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: charts/{name}")

model_keys = list(all_results.keys())

# Chart 1 — Overall accuracy
fig, ax = plt.subplots(figsize=(9,5))
vals   = [all_stats[m]["overall"]["pct"] for m in model_keys]
colors = [MODELS[m]["color"] for m in model_keys]
labels = [MODELS[m]["display"] for m in model_keys]
bars   = ax.bar(labels, vals, color=colors, width=0.5, zorder=3, edgecolor="#0d1117")
for b,v in zip(bars,vals):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+1, f"{v:.1f}%",
            ha="center",va="bottom",fontsize=12,fontweight="bold",color="#f0f6fc")
ax.axhline(10,  color="#ff9a3c",ls="--",lw=1.5,alpha=0.8,label="Random small (10%)")
ax.axhline(4.5, color="#ff6b6b",ls=":",  lw=1.5,alpha=0.8,label="Random large (4.5%)")
ax.set_ylim(0,115); ax.set_ylabel("Accuracy (%)"); ax.legend(framealpha=0.2)
ax.set_title("Overall Accuracy — Model Comparison",fontweight="bold",pad=14)
ax.yaxis.grid(True,zorder=0); ax.set_axisbelow(True)
plt.tight_layout(); save_chart(fig,"01_overall_accuracy.png")

# Chart 2 — By ambiguity
levels = ["clear","paraphrased","ambiguous"]
x = np.arange(len(levels)); w = 0.15
fig, ax = plt.subplots(figsize=(11,6))
for i, mk in enumerate(model_keys):
    vals2 = [all_stats[mk]["by_ambiguity"][lv]["pct"] for lv in levels]
    offset = (i-len(model_keys)/2+0.5)*w
    bars2 = ax.bar(x+offset, vals2, w, label=MODELS[mk]["display"],
                   color=MODELS[mk]["color"], zorder=3, edgecolor="#0d1117")
    for b,v in zip(bars2,vals2):
        ax.text(b.get_x()+b.get_width()/2,b.get_height()+0.5,f"{v:.0f}%",
                ha="center",va="bottom",fontsize=7,color="#f0f6fc")
ax.set_xticks(x); ax.set_xticklabels([l.capitalize() for l in levels],fontsize=11)
ax.set_ylim(0,115); ax.set_ylabel("Accuracy (%)")
ax.set_title("Accuracy by Query Ambiguity Level",fontweight="bold",pad=14)
ax.legend(framealpha=0.2,labelcolor="#f0f6fc",fontsize=9)
ax.yaxis.grid(True,zorder=0); ax.set_axisbelow(True)
plt.tight_layout(); save_chart(fig,"02_accuracy_by_ambiguity.png")

# Chart 3 — By interface size
sizes = ["small","large"]
x3 = np.arange(len(sizes))
fig, ax = plt.subplots(figsize=(9,5))
for i, mk in enumerate(model_keys):
    vals3 = [all_stats[mk]["by_size"][sz]["pct"] for sz in sizes]
    offset = (i-len(model_keys)/2+0.5)*w
    bars3 = ax.bar(x3+offset, vals3, w, label=MODELS[mk]["display"],
                   color=MODELS[mk]["color"], zorder=3, edgecolor="#0d1117")
    for b,v in zip(bars3,vals3):
        ax.text(b.get_x()+b.get_width()/2,b.get_height()+0.5,f"{v:.0f}%",
                ha="center",va="bottom",fontsize=8,color="#f0f6fc")
ax.set_xticks(x3); ax.set_xticklabels(["Small (10 el.)","Large (21-24 el.)"],fontsize=11)
ax.set_ylim(0,115); ax.set_ylabel("Accuracy (%)")
ax.set_title("Accuracy by Interface Complexity",fontweight="bold",pad=14)
ax.legend(framealpha=0.2,labelcolor="#f0f6fc",fontsize=9)
ax.yaxis.grid(True,zorder=0); ax.set_axisbelow(True)
plt.tight_layout(); save_chart(fig,"03_accuracy_by_interface.png")

# Chart 4 — By platform
plats = ["GitHub","AWS"]
x4 = np.arange(len(plats))
fig, ax = plt.subplots(figsize=(9,5))
for i, mk in enumerate(model_keys):
    vals4 = [all_stats[mk]["by_platform"][p]["pct"] for p in plats]
    offset = (i-len(model_keys)/2+0.5)*w
    bars4 = ax.bar(x4+offset, vals4, w, label=MODELS[mk]["display"],
                   color=MODELS[mk]["color"], zorder=3, edgecolor="#0d1117")
    for b,v in zip(bars4,vals4):
        ax.text(b.get_x()+b.get_width()/2,b.get_height()+0.5,f"{v:.0f}%",
                ha="center",va="bottom",fontsize=8,color="#f0f6fc")
ax.set_xticks(x4); ax.set_xticklabels(plats,fontsize=12)
ax.set_ylim(0,115); ax.set_ylabel("Accuracy (%)")
ax.set_title("Accuracy by Platform",fontweight="bold",pad=14)
ax.legend(framealpha=0.2,labelcolor="#f0f6fc",fontsize=9)
ax.yaxis.grid(True,zorder=0); ax.set_axisbelow(True)
plt.tight_layout(); save_chart(fig,"04_accuracy_by_platform.png")

# Chart 5 — Heatmaps
fig, axes = plt.subplots(1, len(model_keys), figsize=(5*len(model_keys),4))
if len(model_keys)==1: axes=[axes]
for ax, mk in zip(axes,model_keys):
    mat = np.array([[all_stats[mk]["cross"][lv][sz]["pct"] for sz in ["small","large"]]
                    for lv in ["clear","paraphrased","ambiguous"]])
    sns.heatmap(mat,ax=ax,annot=True,fmt=".1f",cmap="YlOrRd",
                xticklabels=["Small","Large"],
                yticklabels=["Clear","Paraphrased","Ambiguous"],
                vmin=0,vmax=100,linewidths=0.5,linecolor="#0d1117",
                annot_kws={"size":11,"weight":"bold"})
    ax.set_title(MODELS[mk]["display"],fontweight="bold",color="#f0f6fc")
plt.suptitle("Accuracy Heatmap: Ambiguity x Interface Size (%)",
             fontsize=14,fontweight="bold",color="#f0f6fc",y=1.03)
plt.tight_layout(); save_chart(fig,"05_heatmap.png")

# Chart 6 — Error distribution pies
fig, axes = plt.subplots(1,len(model_keys),figsize=(5*len(model_keys),5))
if len(model_keys)==1: axes=[axes]
ecols = {"clear":"#2ecc71","paraphrased":"#3498db","ambiguous":"#e74c3c"}
for ax, mk in zip(axes,model_keys):
    errors = all_results[mk][~all_results[mk]["is_correct"]]
    if len(errors)==0:
        ax.text(0.5,0.5,"No errors!",ha="center",va="center",fontsize=14,color="#2ecc71",transform=ax.transAxes)
        ax.set_title(MODELS[mk]["display"]); continue
    counts = errors["ambiguity_level"].value_counts()
    wc = [ecols.get(l,"#999") for l in counts.index]
    _, _, autotexts = ax.pie(counts.values,labels=counts.index,colors=wc,
                              autopct="%1.1f%%",startangle=90,
                              wedgeprops={"edgecolor":"#0d1117","linewidth":2})
    for at in autotexts: at.set_fontsize(10); at.set_color("#0d1117"); at.set_fontweight("bold")
    ax.set_title(f"{MODELS[mk]['display']}\n({len(errors)} errors)",
                 fontweight="bold",color="#f0f6fc")
plt.suptitle("Error Distribution by Ambiguity Level",fontsize=13,fontweight="bold",color="#f0f6fc")
plt.tight_layout(); save_chart(fig,"06_error_distribution.png")

# Chart 7 — Radar
cats = ["Clear","Paraphrased","Ambiguous","Small UI","Large UI"]
N = len(cats); angles = [n/float(N)*2*np.pi for n in range(N)]+[0]
fig, ax = plt.subplots(figsize=(7,7),subplot_kw=dict(polar=True))
ax.set_facecolor("#161b22"); fig.patch.set_facecolor("#0d1117")
for mk in model_keys:
    s = all_stats[mk]
    vals7 = [s["by_ambiguity"]["clear"]["pct"],s["by_ambiguity"]["paraphrased"]["pct"],
             s["by_ambiguity"]["ambiguous"]["pct"],s["by_size"]["small"]["pct"],
             s["by_size"]["large"]["pct"],s["by_ambiguity"]["clear"]["pct"]]
    ax.plot(angles,vals7,"o-",lw=2,color=MODELS[mk]["color"],label=MODELS[mk]["display"])
    ax.fill(angles,vals7,alpha=0.1,color=MODELS[mk]["color"])
ax.set_xticks(angles[:-1]); ax.set_xticklabels(cats,size=10,color="#f0f6fc")
ax.set_ylim(0,100); ax.grid(color="#30363d",lw=0.8); ax.spines["polar"].set_color("#30363d")
ax.legend(loc="upper right",bbox_to_anchor=(1.4,1.1),framealpha=0.2,labelcolor="#f0f6fc",fontsize=9)
ax.set_title("Model Performance Radar",pad=20,fontweight="bold",color="#f0f6fc",size=13)
plt.tight_layout(); save_chart(fig,"07_radar.png")

# Chart 8 — Per-task line
fig, ax = plt.subplots(figsize=(16,5))
for mk in model_keys:
    ta = all_results[mk].groupby("task_id")["is_correct"].mean()*100
    ax.plot(ta.index,ta.values,marker="o",markersize=3,lw=1.5,
            color=MODELS[mk]["color"],label=MODELS[mk]["display"],alpha=0.85)
ax.set_xlabel("Task ID"); ax.set_ylabel("Accuracy (%)")
ax.set_title("Per-Task Accuracy Across All Models",fontweight="bold",pad=14)
ax.legend(framealpha=0.2,labelcolor="#f0f6fc",fontsize=9)
ax.yaxis.grid(True,alpha=0.4); ax.set_axisbelow(True); ax.set_xlim(1,60); ax.set_ylim(-5,105)
plt.tight_layout(); save_chart(fig,"08_per_task_accuracy.png")

print("\n✓ All 8 charts saved to charts/ folder")

In [ ]:
# ── CELL 9: Statistical significance tests (McNemar's) ──────────────────────
from scipy.stats import chi2_contingency
try:
    from statsmodels.stats.contingency_tables import mcnemar as mcnemar_test
    has_statsmodels = True
except:
    has_statsmodels = False
    print("Note: statsmodels not available, using chi2 approximation")

print("McNemar's Test — Pairwise Model Comparison")
print("="*55)
print(f"  {'Pair':<35}  {'p-value':>10}  {'Result'}")
print("  "+"-"*60)

keys = list(all_results.keys())
sig_results = []
for i in range(len(keys)):
    for j in range(i+1, len(keys)):
        a = all_results[keys[i]]["is_correct"].values
        b = all_results[keys[j]]["is_correct"].values
        both_right = int(np.sum(a & b))
        a_only     = int(np.sum(a & ~b))
        b_only     = int(np.sum(~a & b))
        both_wrong = int(np.sum(~a & ~b))
        table = [[both_right, a_only],[b_only, both_wrong]]
        pair_name = f"{MODELS[keys[i]]['display']} vs {MODELS[keys[j]]['display']}"
        try:
            if has_statsmodels:
                res = mcnemar_test(table, exact=False, correction=True)
                p = res.pvalue
            else:
                n = a_only + b_only
                p = 1.0 if n == 0 else float(2 * min(a_only, b_only) / n)
            sig = "SIGNIFICANT *" if p < 0.05 else "not significant"
            print(f"  {pair_name:<35}  p={p:.4f}      {sig}")
            sig_results.append({"pair": pair_name, "p_value": round(p,4), "significant": p<0.05})
        except Exception as e:
            print(f"  {pair_name:<35}  ERROR: {e}")

print("\n  * p < 0.05 = statistically significant difference")

In [ ]:
# ── CELL 10: Export all results ─────────────────────────────────────────────
import os
from google.colab import files as colab_files

os.makedirs("results", exist_ok=True)
os.makedirs("charts", exist_ok=True)

# Save per-model CSVs
for mk in all_results:
    path = f"results/results_{mk}.csv"
    all_results[mk].to_csv(path, index=False, encoding="utf-8")
    print(f"  Saved: {path}")

# Save combined CSV
base = list(all_results.values())[0][
    ["task_id","platform","query","ambiguity_level",
     "interface_size","ui_elements","correct_answer"]].copy()
for mk in all_results:
    short = MODELS[mk]["display"].replace(" ","_")
    base[f"{short}_prediction"] = all_results[mk]["predicted"].values
    base[f"{short}_correct"]    = all_results[mk]["is_correct"].values
base.to_csv("results/combined_all_models.csv", index=False, encoding="utf-8")
print("  Saved: results/combined_all_models.csv")

# Save report
lines = ["="*65,"  LLM UI ELEMENT SELECTION — EXPERIMENT REPORT",
         f"  Generated: {__import__('datetime').datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
         "="*65,""]
lines += ["4.2 OVERALL ACCURACY","─"*65]
for mk in all_stats:
    s = all_stats[mk]["overall"]
    lines.append(f"  {MODELS[mk]['display']:<18}: {s['pct']}%  ({s['correct']}/{s['total']})")
lines += ["","4.3 BY AMBIGUITY LEVEL","─"*65]
for lv in ["clear","paraphrased","ambiguous"]:
    row_str = f"  {lv.capitalize():<14}: "
    for mk in all_stats:
        s = all_stats[mk]["by_ambiguity"][lv]
        row_str += f"{MODELS[mk]['display']}={s['pct']}%  "
    lines.append(row_str)
lines += ["","4.4 BY INTERFACE SIZE","─"*65]
for sz in ["small","large"]:
    row_str = f"  {sz.capitalize():<14}: "
    for mk in all_stats:
        s = all_stats[mk]["by_size"][sz]
        row_str += f"{MODELS[mk]['display']}={s['pct']}%  "
    lines.append(row_str)
lines += ["","4.5 SIGNIFICANCE TESTS","─"*65]
for r in sig_results:
    lines.append(f"  {r['pair']}: p={r['p_value']}  ({'SIGNIFICANT' if r['significant'] else 'not significant'})")
lines.append("\n"+"="*65)
with open("reports/experiment_report.txt","w",encoding="utf-8") as f:
    f.write("\n".join(lines))
print("  Saved: reports/experiment_report.txt")

# Zip everything and download
import zipfile
with zipfile.ZipFile("experiment_results.zip","w") as zf:
    for folder in ["results","charts","reports"]:
        for fn in os.listdir(folder):
            zf.write(f"{folder}/{fn}")

print("\n✓ Downloading experiment_results.zip...")
colab_files.download("experiment_results.zip")
print("✓ Done! Check your downloads folder.")